Super ! Bon çà reste des resultats fragiles vue la taille du jeu de données, mais on semble quand même obtenir une capacité de traitement bien supérieure aux modèles précédents. On note en particulier la capacité à distinguer les phrases possédant des mots identiques mais dans un ordre différent.

# Experimentations sur un Jeu de Données Réel : IMDB

Pour terminer ce TP, on s'intéresse à l'adaptation du modèle DistillBert utilisé ci-dessus, sur un jeu de données  beaucoup plus conséquent: le corpus IMDB (base de données de commentaires sur des films). Pour cette partie, il est largement conseillé de travailler sur GPU.

In [1]:
import datasets
dataset = datasets.load_dataset("imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [2]:

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [4]:
train_fraction=0.2
train_dataset=dataset["train"]
train_dataset = train_dataset.train_test_split(test_size=1-train_fraction)["train"]
test_fraction=0.002
validation_dataset=dataset["test"]
validation_dataset = validation_dataset.train_test_split(test_size=1-test_fraction)["train"]

from transformers import AutoTokenizer, DistilBertForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=True)


train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.features.keys(), load_from_cache_file=True)
print(train_dataset)

validation_dataset = validation_dataset.map(preprocess_function, batched=True, remove_columns=validation_dataset.features.keys(), load_from_cache_file=True)
print(validation_dataset)



config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 5000
})


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 50
})


In [6]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
import torch
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler
from transformers import DataCollatorWithPadding
# Let's define a basic data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
pad_idx=tokenizer.pad_token_id  # On met à jour l'index de padding et on recrée les dataloaders
batchsize=32
megamul=4
# Fix the typo for megamul
megabatch_mul = megamul # This line will make megabatch_mul available. The user likely intended to use `megamul`
# Define data_collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
pad_idx=tokenizer.pad_token_id  # On met à jour l'index de padding et on recrée les dataloaders
batchsize=32
megamul=4
train_sampler = RandomSampler(train_dataset) # Temporary replacement

train_loader=DataLoader(train_dataset, batch_size=batchsize, sampler=train_sampler, collate_fn=data_collator, pin_memory=True, shuffle=False, num_workers=0)

for data in train_loader:
    print(data)
    print(tokenizer.batch_decode(data["input_ids"]))
    break

# test_sampler = LengthGroupedSampler(batchsize, validation_dataset, megamul) # Original line with corrected typo
test_sampler = SequentialSampler(validation_dataset) # Temporary replacement

test_loader=DataLoader(validation_dataset, batch_size=batchsize, sampler=test_sampler, collate_fn=data_collator, pin_memory=True, shuffle=False, num_workers=0)


for data in test_loader:
    print(data)
    print(tokenizer.batch_decode(data["input_ids"]))
    break

{'input_ids': tensor([[  101,  6203,  8141,  ...,     0,     0,     0],
        [  101,  2023,  2143,  ...,  1005,  1055,   102],
        [  101,  2045,  2020,  ...,     0,     0,     0],
        ...,
        [  101,  1037,  5122,  ...,     0,     0,     0],
        [  101,  6071, 17084,  ...,     0,     0,     0],
        [  101,  2000,  2023,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}
["[CLS] dear readers, < br / > < br / > the final battle between the rebellion and empire. the second death star is nearing completion and when it is

In [7]:
def train_test(trainloader, testloader, model, loss_function, optimizer, epochs, clip=-1, test_rate=1):
  it=0

  for epoch in range(epochs):
      epoch_loss = 0
      epoch_accuracy = 0

      nb_samples=0
      t = tqdm(iter(trainloader), total=len(trainloader), dynamic_ncols=True, position=0)
      train_loss_log = tqdm(total=0, position=4, bar_format='{desc}')
      test_log = tqdm(total=0, position=2, bar_format='{desc}')
      accuracy_log = tqdm(total=0, position=3, bar_format='{desc}')

      # Get model's device once at the beginning of each epoch
      model_device = next(model.parameters()).device

      for batch in t:
          it+=1
          if it%test_rate==0:
            test(testloader, model, loss_function, epoch,(test_log,accuracy_log))
          model.train()
          optimizer.zero_grad()
          #print("text shape ",batch.text.T.shape)
          prediction = model(batch["input_ids"].to(model_device))
          if not isinstance(prediction,torch.Tensor):
                prediction = prediction["logits"]
          #print(prediction)
          loss = loss_function(prediction, batch["labels"].to(model_device))
          train_loss_log.set_description_str("loss train {:.3f} ".format(loss.item()))
          loss.backward()
          if clip>0:
              torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
          optimizer.step()
          nb_samples+=prediction.shape[0]
          epoch_loss+=loss.item()*prediction.shape[0]
          preds=(prediction[:,1]>prediction[:,0])*1.0
          accuracy=(preds==batch["labels"].to(model_device)).sum()
          epoch_accuracy+=accuracy.item()
      print('train loss on epoch {} : {:.3f}'.format(epoch, epoch_loss/nb_samples))
      print('train accuracy on epoch {}: {:.3f}'.format(epoch, epoch_accuracy/nb_samples))

def test(testloader, model, loss_function, epoch, descs):
      model.eval()
      test_loss = 0
      test_accuracy = 0
      nb_samples=0
      accuracy=0
      t = tqdm(iter(testloader), total=len(testloader), position=1, leave=False)
      tl,al=descs

      # Get model's device
      model_device = next(model.parameters()).device

      for batch in t:
          #print("test ",batch)
          with torch.no_grad():
              optimizer.zero_grad()
              inputs=batch["input_ids"].to(model_device)
              prediction = model(inputs)
              if not isinstance(prediction,torch.Tensor):
                    prediction = prediction["logits"]
              loss = loss_function(prediction, batch["labels"].to(model_device))
              nb_samples+=prediction.shape[0]
              test_loss+=loss.item()*prediction.shape[0]
              #print(batch_decode(batch["input_ids"]))
              preds=(prediction[:,1]>prediction[:,0])*1.0
              accuracy=(preds==batch["labels"].to(model_device)).sum()
              test_accuracy+=accuracy.item()

In [18]:
import torch
from torch.optim import Adam
from tqdm.auto import tqdm
from transformers import DistilBertForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Load model
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
).to(device)

optimizer = Adam(model.parameters(), lr=2e-5)

epochs = 3

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):

        batch = {k: v.to(device) for k, v in batch.items()}

        # récupérer automatiquement la colonne label
        label_key = [k for k in batch.keys() if k not in ["input_ids","attention_mask"]][0]
        labels = batch[label_key]

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch", epoch+1, "loss:", total_loss/len(train_loader))


    # evaluation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():

        for batch in test_loader:

            batch = {k: v.to(device) for k, v in batch.items()}

            label_key = [k for k in batch.keys() if k not in ["input_ids","attention_mask"]][0]
            labels = batch[label_key][:,0]

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )

            preds = torch.argmax(outputs.logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    print("Accuracy:", correct/total)

cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/157 [00:00<?, ?it/s]

ValueError: Expected input batch_size (32) to match target batch_size (16384).

In [19]:
batch = next(iter(train_loader))
for k,v in batch.items():
    print(k, v.shape)

input_ids torch.Size([32, 512])
token_type_ids torch.Size([32, 512])
attention_mask torch.Size([32, 512])


In [21]:
import torch
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler
from datasets import load_dataset
from transformers import AutoTokenizer, DistilBertForSequenceClassification, DataCollatorWithPadding
from torch.optim import Adam

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

dataset = load_dataset("imdb")

train_fraction = 0.2
train_dataset = dataset["train"].train_test_split(test_size=1-train_fraction)["train"]

test_fraction = 0.002
validation_dataset = dataset["test"].train_test_split(test_size=1-test_fraction)["train"]

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    encodings = tokenizer(
        examples["text"],
        truncation=True,
        padding=True,
        max_length=256
    )
    encodings["labels"] = examples["label"]  # <- indispensable pour DistilBERT
    return encodings

train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["text"],   # on supprime seulement text
    load_from_cache_file=True
)

validation_dataset = validation_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["text"],
    load_from_cache_file=True
)


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=RandomSampler(train_dataset),
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=2
)

val_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    sampler=SequentialSampler(validation_dataset),
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=2
)


model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
model.to(device)

optimizer = Adam(model.parameters(), lr=2e-5)
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} | Train loss: {total_loss/len(train_loader):.4f}")

    # Évaluation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == batch["labels"]).sum().item()
            total += batch["labels"].size(0)
    print(f"Validation Accuracy: {correct/total:.4f}")

Using device: cuda


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1 | Train loss: 0.3868
Validation Accuracy: 0.9000
Epoch 2 | Train loss: 0.2018
Validation Accuracy: 0.9200
Epoch 3 | Train loss: 0.1086
Validation Accuracy: 0.9200
